In [61]:
from __future__ import annotations
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import gc
import json
import time
import warnings
import os
from pathlib import Path
from arch import arch_model
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================

PROCESSED_DIR = Path("../processed")
N_FOLDS       = 5

INPUT_START = 0
INPUT_END   = 480
TARGET_END  = 600
TARGET_LEN  = TARGET_END - INPUT_END  

EPS      = 1e-12
RET_CLIP = 0.05

GARCH_P = 1
GARCH_Q = 1
DIST    = "normal"

MIN_RET_STD    = 1e-8
SCALE          = 100.0 
RV_EVAL_FLOOR  = 1e-4 

N_JOBS = max(1, min((os.cpu_count() or 2) - 1, 8))

In [62]:
# =============================================================================
# METRICS
# =============================================================================

def compute_metrics(
    pred_log_rv: np.ndarray,
    true_log_rv: np.ndarray,
) -> dict:
    pred_log_rv = np.clip(pred_log_rv, -20, 5)
    pred_rv = np.clip(np.exp(pred_log_rv), EPS, None)
    true_rv = np.clip(np.exp(true_log_rv), EPS, None)

    ratio = true_rv / pred_rv
    resid = pred_rv - true_rv

    return {
        "QLIKE":      float(np.mean(ratio - np.log(np.clip(ratio, EPS, None)) - 1.0)),
        "RMSPE%":     float(np.sqrt(np.mean((resid / true_rv) ** 2)) * 100),
        "MAPE%":      float(np.mean(np.abs(resid) / true_rv) * 100),
        "RMSE":       float(np.sqrt(np.mean(resid ** 2))),
        "MSE_rv":     float(np.mean(resid ** 2)),
        "MAE_log":    float(np.mean(np.abs(pred_log_rv - true_log_rv))),
        "MSE_log_rv": float(np.mean((pred_log_rv - true_log_rv) ** 2)),
    }


def log_metrics(m: dict, indent: int = 4) -> None:
    pad = " " * indent
    for k, v in m.items():
        print(f"{pad}{k:12s}: {v:.6f}")

In [63]:
# =============================================================================
# RETURNS
# =============================================================================

def safe_log_returns(px: np.ndarray) -> np.ndarray:
    px = np.clip(np.asarray(px, dtype=np.float64), EPS, None)

    log_px = np.log(px)

    ret = np.empty_like(log_px)
    ret = np.zeros_like(log_px)
    ret[1:] = np.diff(log_px)

    ret = np.clip(ret, -RET_CLIP, RET_CLIP)

    return np.nan_to_num(ret, nan=np.nan)


def realised_vol(r: np.ndarray) -> float:
    return float(np.sqrt(np.sum(np.square(r))))

# =============================================================================
# PER-OBSERVATION GARCH FORECAST
# =============================================================================

def garch_forecast_one(inp_ret: np.ndarray) -> float | None:
    """
    Fit GARCH(1,1) on inp_ret (480 points), return predicted RV
    over the next TARGET_LEN steps.

    Returns None on failure so the caller can fall back to a baseline.
    """
    if np.std(inp_ret) < MIN_RET_STD:
        return None

    y = inp_ret * SCALE     # work in percent-return space

    try:
        am  = arch_model(y, mean="Zero", vol="Garch",
                         p=GARCH_P, q=GARCH_Q, dist=DIST, rescale=False)
        fit = am.fit(disp="off", show_warning=False,
                     options={"maxiter": 50}, tol=1e-4)

        # arch forecast() handles multi-step correctly
        fc     = fit.forecast(horizon=TARGET_LEN, reindex=False)
        var_fc = np.asarray(fc.variance.values.flatten(), dtype=np.float64)
        var_fc = np.maximum(var_fc, 0.0)

        if len(var_fc) < TARGET_LEN or not np.isfinite(var_fc).all():
            return None

        # var_fc is in (percent-return)^2 units → unscale to decimal-return units
        pred_rv = float(np.sqrt(np.sum(var_fc))) / SCALE
        return pred_rv if np.isfinite(pred_rv) and pred_rv > 0 else None

    except Exception:
        return None

# =============================================================================
# BASELINE: scaled input RV  (fast fallback for failed fits)
# =============================================================================
def baseline_rv(inp_ret: np.ndarray, fut_len: int) -> float:
    rv_in = realised_vol(inp_ret)

    if len(inp_ret) == 0:
        return EPS

    return rv_in * np.sqrt(fut_len / len(inp_ret))


In [64]:
# =============================================================================
# PROCESS ONE (stock, time_id) TASK
# =============================================================================

def _process_one(stock_id, time_id, grp: pd.DataFrame) -> dict | None:

    grp = grp.sort_values("seconds_in_bucket").copy()

    px = grp["wap"].to_numpy(dtype=np.float64)

    grp["ret"] = safe_log_returns(px)


    inp = grp.loc[
        grp["seconds_in_bucket"] < INPUT_END,
        "ret"
    ].to_numpy(dtype=np.float64)

    fut = grp.loc[
        (grp["seconds_in_bucket"] >= INPUT_END)
        & (grp["seconds_in_bucket"] < TARGET_END),
        "ret"
    ].to_numpy(dtype=np.float64)
    
    inp_clean = inp[~np.isnan(inp)]
    fut_clean = fut[~np.isnan(fut)]

    if len(inp_clean) < 5 or len(fut_clean) < 1:
        return None

    rv_in = max(realised_vol(inp_clean), MIN_RET_STD)
    rv_fut = max(realised_vol(fut_clean), RV_EVAL_FLOOR)

    
    if rv_in < MIN_RET_STD:
        pred_rv = baseline_rv(inp_clean, len(fut_clean))
        fallback = True
    else:
        pred_rv = garch_forecast_one(inp_clean)
        fallback = pred_rv is None

        if fallback:
            pred_rv = baseline_rv(inp_clean, len(fut_clean))
    pred_rv = max(pred_rv, EPS)

    log_rv_in = np.log(rv_in + EPS)
    log_rv_fut = np.log(rv_fut + EPS)
    log_rv_pred = np.log(pred_rv + EPS)

    return {
        "stock_id": stock_id,
        "time_id": time_id,
        "true_rv": rv_fut,
        "pred_rv": pred_rv,
        "true_log_rv": log_rv_fut,
        "pred_log_rv": log_rv_pred,
        "anchor_log_rv": log_rv_in,
        "true_log_rv_diff": log_rv_fut - log_rv_in,
        "pred_log_rv_diff": log_rv_pred - log_rv_in,
        "fallback": int(fallback),
        "n_input_obs": len(inp),
        "n_future_obs": len(fut),
    }

# =============================================================================
# PROCESS ONE ROW-GROUP
# =============================================================================

def process_row_group(df: pd.DataFrame, n_jobs: int = N_JOBS) -> list[dict]:
    df = df.sort_values(["stock_id", "time_id", "seconds_in_bucket"])

    tasks = []
    for (stock_id, time_id), grp in df.groupby(
        ["stock_id", "time_id"],
        sort=False
    ):
        tasks.append((stock_id, time_id, grp.copy()))

    if not tasks:
        return []

    results = Parallel(n_jobs=n_jobs, backend="loky", batch_size=32)(
        delayed(_process_one)(sid, tid, px) for sid, tid, px in tasks
    )

    return [r for r in results if r is not None]

# =============================================================================
# RUN ONE FOLD
# =============================================================================

def run_fold(fold_id: int, use_cache: bool = True) -> dict:
    print(f"\n{'='*70}")
    print(f"FOLD {fold_id}")
    print(f"{'='*70}")

    out_dir = Path(".")
    pred_path       = out_dir / f"garch_predictions_fold{fold_id}.csv"
    per_stock_path  = out_dir / f"garch_per_stock_fold{fold_id}.csv"

    if use_cache and pred_path.exists() and per_stock_path.exists():
        print(f"  Loading cached results for fold {fold_id}...")
        pred_df = pd.read_csv(pred_path)
        m = compute_metrics(pred_df["pred_log_rv"].values, pred_df["true_log_rv"].values)

        last_rv = pred_df["anchor_log_rv"].values
        m_last = compute_metrics(last_rv, pred_df["true_log_rv"].values)

        fallback_pct = pred_df["fallback"].mean() * 100
        n_raw = len(pred_df)
        n_kept = n_raw
        print(f"  Cached rows: {n_kept}, Fallback: {fallback_pct:.1f}%")
        return {
            "fold":         fold_id,
            "rows_raw":     n_raw,
            "rows_eval":    n_kept,
            "fallback_pct": fallback_pct,
            "elapsed_min":  0.0,
            **{f"garch_{k}":  v for k, v in m.items()},
            **{f"last_{k}":   v for k, v in m_last.items()},
        }

    test_path = PROCESSED_DIR / f"fold_{fold_id}" / "test.parquet"
    t0 = time.perf_counter()

    pf = pq.ParquetFile(test_path)
    rows = []

    print(f"  N_JOBS: {N_JOBS}")
    for rg in range(pf.num_row_groups):
        print(f"  Row group {rg+1}/{pf.num_row_groups}", end="\r")
        df = pf.read_row_group(rg).to_pandas()
        rows.extend(process_row_group(df, n_jobs=N_JOBS))
        del df
        gc.collect()

    print()
    pred_df = pd.DataFrame(rows)
    if pred_df.empty:
        print("  No predictions — check data format.")
        return {}

    elapsed = (time.perf_counter() - t0) / 60.0

    n_raw = len(pred_df)
    n_kept = len(pred_df)

    m = compute_metrics(pred_df["pred_log_rv"].values, pred_df["true_log_rv"].values)
    last_rv = pred_df["anchor_log_rv"].values
    m_last = compute_metrics(last_rv, pred_df["true_log_rv"].values)
    fallback_pct = pred_df["fallback"].mean() * 100

    pred_df.to_csv(pred_path, index=False)

    per_stock_df = pd.DataFrame([
        {"stock_id": sid, **sm} for sid, sm in 
        {sid: compute_metrics(grp["pred_log_rv"].values, grp["true_log_rv"].values)
         for sid, grp in pred_df.groupby("stock_id") if len(grp) >= 5}.items()
    ])
    if not per_stock_df.empty and "RMSPE%" in per_stock_df.columns:
        per_stock_df = per_stock_df.sort_values("RMSPE%")
    per_stock_df.to_csv(per_stock_path, index=False)

    print(f"\n  Saved -> {pred_path}, {per_stock_path}")

    return {
        "fold":         fold_id,
        "rows_raw":     n_raw,
        "rows_eval":    n_kept,
        "fallback_pct": fallback_pct,
        "elapsed_min":  elapsed,
        **{f"garch_{k}":  v for k, v in m.items()},
        **{f"last_{k}":   v for k, v in m_last.items()},
    }

In [65]:
# =============================================================================
# MAIN
# =============================================================================

def main():
    print("=" * 70)
    print("GARCH(1,1) RV Forecasting — Nested CV evaluation")
    print(f"  Input  : seconds {INPUT_START}–{INPUT_END-1}  ({INPUT_END} points)")
    print(f"  Target : seconds {INPUT_END}–{TARGET_END-1}  ({TARGET_LEN} points)")
    print(f"  Model  : GARCH({GARCH_P},{GARCH_Q}) fit per (stock, time_id)")
    print(f"  Filter : true_rv > {RV_EVAL_FLOOR}")
    print(f"  Folds  : {N_FOLDS}")
    print("=" * 70)

    all_results = []
    t_total = time.perf_counter()

    for fold_id in range(N_FOLDS):
        res = run_fold(fold_id, use_cache=True)
        if res:
            all_results.append(res)
        gc.collect()

    total_min = (time.perf_counter() - t_total) / 60.0
    if not all_results:
        print("No results produced.")
        return

    results_df = pd.DataFrame(all_results)
    print("\nNESTED CV SUMMARY")
    print("=" * 70)
    print(results_df)

main()

GARCH(1,1) RV Forecasting — Nested CV evaluation
  Input  : seconds 0–479  (480 points)
  Target : seconds 480–599  (120 points)
  Model  : GARCH(1,1) fit per (stock, time_id)
  Filter : true_rv > 0.0001
  Folds  : 5

FOLD 0
  N_JOBS: 8
  Row group 112/112

  Saved -> garch_predictions_fold0.csv, garch_per_stock_fold0.csv

FOLD 1
  N_JOBS: 8
  Row group 112/112

  Saved -> garch_predictions_fold1.csv, garch_per_stock_fold1.csv

FOLD 2
  N_JOBS: 8
  Row group 112/112

  Saved -> garch_predictions_fold2.csv, garch_per_stock_fold2.csv

FOLD 3
  N_JOBS: 8
  Row group 112/112

  Saved -> garch_predictions_fold3.csv, garch_per_stock_fold3.csv

FOLD 4
  N_JOBS: 8
  Row group 112/112

  Saved -> garch_predictions_fold4.csv, garch_per_stock_fold4.csv

NESTED CV SUMMARY
   fold  rows_raw  rows_eval  fallback_pct  elapsed_min  garch_QLIKE  \
0     0     85786      85786           0.0     7.630051     0.056680   
1     1     85790      85790           0.0     8.594313     0.056715   
2     2     8